# Phase 11 Part 1A — Historical Protocol Reconstruction + CLEAN-v3 Failure-Definition Freeze

**Model:** Gemini 3.1 Pro High · **Date:** 2026-09-21
**Manifest:** `phase4_v3` · **Taxonomy:** `kub_size_clean_v3_v1`
**Phase:** 11 (Part 1A) — Baseline Failure Analysis

---

> **PROTOCOL / DEFINITION PHASE ONLY.** Part 1A does **not** execute the full Phase-11
> failure analysis. It reconstructs the historical Phase-11 methodology and freezes the
> clean-v3 failure-analysis definitions *before* any detailed inspection.
>
> Forbidden here: training, checkpoint/history modification, threshold tuning, winner or
> reference-model selection, Phase 12, TEST access.
>
> **No model is called a winner. No reference model is selected.**


## 2. Scope and Governance

| Rule | Status |
|---|---|
| Training epochs | **0** |
| Optimizer steps | **0** |
| Checkpoint writes | **0** |
| TEST image/mask/content reads | **0** |
| TEST metrics / visualizations | **0** |
| Winner declared | **NO** |
| Reference model selected | **NO** |
| New numerical thresholds introduced | **0** |
| Phase 12 started | **NO** |
| Full Phase-11 failure analysis | **NOT STARTED** |


## 3. Phase-10 Frozen Evidence (inputs)

Four frozen clean-v3 baselines. Values are single-seed descriptive VAL evidence.


In [1]:
%matplotlib inline
import sys, json
from pathlib import Path
import numpy as np, pandas as pd
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

PROJECT_ROOT = Path(r'D:\Softfors\jup')
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / 'notebooks' / 'phasewise' / 'phase_11_failure_analysis' / 'code'))
import phase11_failure_protocol as p

rows=[]
for k in p.CLEAN_V3_MODELS:
    m = p.CLEAN_V3_METRICS[k]
    rows.append({'model': p.MODEL_DISPLAY[k], 'best_epoch': m['best_epoch'],
                 'dice': m['dice'], 'iou': m['iou'],
                 'precision': m['precision'], 'recall': m['recall']})
P10 = pd.DataFrame(rows)
display(P10)
print('Manifest: phase4_v3 | Taxonomy: kub_size_clean_v3_v1')
print('These are descriptive single-seed VAL results - NOT a ranking, NO winner.')


,model,best_epoch,dice,iou,precision,recall
0,UNet,44,0.740613,0.625382,0.770719,0.783944
1,Attention U-Net,54,0.752513,0.634939,0.772914,0.792225
2,UNet++,70,0.764160,0.652544,0.799683,0.791238
3,DeepLabV3+,52,0.750914,0.646049,0.794333,0.782298


Manifest: phase4_v3 | Taxonomy: kub_size_clean_v3_v1
These are descriptive single-seed VAL results - NOT a ranking, NO winner.


In [2]:
# Phase-10 paired VAL evidence is reused READ-ONLY (no re-inference in Part 1A)
pv = p.verify_paired_table()
print('=== Phase-10 paired VAL table (read-only) ===')
for k,v in pv.items(): print(f'  {k}: {v}')
assert pv['rows']==280 and pv['all_model_columns'] and not pv['any_nan']
assert pv['acquisition_counts']=={'ROI':177,'FULL_FIELD':103}
print('\nPAIRED EVIDENCE REUSED - PASS (280 cases, all four models, no rerun needed)')


=== Phase-10 paired VAL table (read-only) ===
  rows: 280
  unique_samples: 280
  all_model_columns: True
  any_nan: False
  category_counts: {'Bladder_Stone': 144, 'Kidney_Stone': 112, 'Ureteric_Stone': 24}
  acquisition_counts: {'ROI': 177, 'FULL_FIELD': 103}

PAIRED EVIDENCE REUSED - PASS (280 cases, all four models, no rerun needed)


## 4. Historical Phase-11 Source Inventory


In [3]:
inv = p.build_source_inventory()
display(inv[['precedence','source_kind','path','exists']])
print('\nAll sources present:', bool(inv.exists.all()))


,precedence,source_kind,path,exists
0,1,executing analysis code,scripts/analysis/analyze_phase11_failures.py,True
1,1,executing selection code,scripts/generate_phase11_failures.py,True
2,2,persisted strata definitions,reports/audits/phase11_failure_strata_definitions.json,True
3,2,persisted audit,reports/audits/phase11_kub_failure_analysis.json,True
4,3,executed historical notebook,notebooks/phasewise/phase_11_failure_analysis/phase11_kub_failure_analysis_and_tiny_stone_errors.ipynb,True
5,4,historical README,notebooks/phasewise/phase_11_failure_analysis/README.md,True
6,2,per-image table,reports/tables/phase11_kub_failure_analysis_per_image.csv,True
7,2,per-component table,reports/tables/phase11_kub_failure_analysis_per_component.csv,True
8,2,multicomponent table,reports/tables/phase11_kub_multicomponent_failure.csv,True
9,2,size failure summary,reports/tables/phase11_kub_size_failure_summary.csv,True



All sources present: True


## 5. Historical Model / Split Scope


In [4]:
hist = json.loads((p.AUDITS/'phase11_historical_failure_protocol_reconstruction.json').read_text(encoding='utf-8'))
print('Historical Phase 11 found:', hist['historical_phase11_found'])
print('Primary model   :', hist['historical_model_scope']['primary_model'],
      f"({hist['historical_model_scope']['primary_experiment']})")
print('Comparator      :', hist['historical_model_scope']['secondary_comparator'])
print('Split           :', hist['historical_split'])
print('Threshold       :', hist['historical_threshold'])
print('Units           :', ', '.join(hist['historical_failure_units']))
print()
print('NOTE: historical was UNet++-PRIMARY. Part 1A does NOT adopt a clean-v3 reference model;')
print('      that decision is deferred to the supervisor (single-seed VAL is insufficient).')


Historical Phase 11 found: True
Primary model   : UNet++ (phase10_kub_unetpp_seed42_512)
Comparator      : DeepLabV3+
Split           : VAL_ONLY (283 KUB VAL cases, manifest phase4_v2)
Threshold       : 0.5
Units           : case-level (per_image), component/object-level (per_component), patient-level

NOTE: historical was UNet++-PRIMARY. Part 1A does NOT adopt a clean-v3 reference model;
      that decision is deferred to the supervisor (single-seed VAL is insufficient).


## 6. Historical Failure Definitions

The failure-definition matrix uses `VERIFIED` / `NOT_FOUND` / `AMBIGUOUS`.


In [5]:
mx = p.build_failure_matrix()
display(mx[['concept','unit','metric','threshold','status']])
print()
print('Status counts:', mx.status.value_counts().to_dict())
print()
print('Historical failure labels:', ', '.join(hist['historical_failure_definitions']['labels']))


,concept,unit,metric,threshold,status
0,low Dice,case,dice,dice <= 0.5461063385009766,VERIFIED
1,low IoU,case,iou,NOT_FOUND,NOT_FOUND
2,false-negative dominant,case,recall,NOT_FOUND,NOT_FOUND
3,false-positive dominant,case,fp_eval_components,fp_eval_components >= 4.0,VERIFIED
4,empty prediction,case,empty_prediction,prediction foreground == 0,VERIFIED
5,object miss,component,missed_eval_components / detected,overlap < 1 px => missed,VERIFIED
6,partial object miss,case,detection_class = PARTIAL_DETECTION,0 < detected < total,VERIFIED
7,under-segmentation,case/pixel,n/a,NOT_FOUND,NOT_FOUND
8,over-segmentation,case/pixel,n/a,NOT_FOUND,NOT_FOUND
9,boundary error,case,hd95 / hd95_valid,hd95 >= 83.07246563841615 (valid only),VERIFIED



Status counts: {'VERIFIED': 14, 'NOT_FOUND': 4}

Historical failure labels: EMPTY_PREDICTION, LOW_DICE_DECILE, HIGH_HD95_DECILE, HIGH_FP_DECILE, MISSED_TINY_COMPONENT, MISSED_SMALL_COMPONENT, PARTIAL_MULTI_COMPONENT_DETECTION, COMPLETE_MULTI_COMPONENT_MISS


In [6]:
print('=== Historical numeric thresholds (DESCRIPTIVE strata, NOT tuning thresholds) ===')
for k,v in hist['historical_numeric_thresholds'].items():
    if k=='caveat': continue
    print(f"  {k}: cutoff={v['cutoff']} ({v['comparison']}), n={v['n']}")
print()
print('Caveat:', hist['historical_numeric_thresholds']['caveat'])
print()
print('These cutoffs are HISTORICAL (phase4_v2, 283 VAL cases). They are NOT carried over')
print('as clean-v3 numbers; only the METHOD (own-distribution bottom decile) is inherited.')


=== Historical numeric thresholds (DESCRIPTIVE strata, NOT tuning thresholds) ===


  low_dice_decile: cutoff=0.5461063385009766 (dice <= cutoff), n=29
  high_hd95_decile: cutoff=83.07246563841615 (valid hd95 >= cutoff), n=28
  high_fp_decile: cutoff=4.0 (fp evaluation components >= cutoff), n=39

Caveat: descriptive failure-analysis strata; NOT tuning/deployment thresholds

These cutoffs are HISTORICAL (phase4_v2, 283 VAL cases). They are NOT carried over
as clean-v3 numbers; only the METHOD (own-distribution bottom decile) is inherited.


## 7. Historical Qualitative-Selection Protocol


In [7]:
hc = p.build_hard_case_protocol()
display(hc)
print()
print('Key point: this cohort IS selected by failure status (prediction-dependent), unlike the')
print('Phase-10 qualitative gallery which is prediction-independent. That is intentional for')
print('failure analysis, but it must never be presented as a representative sample.')
print()
print('Part 1A freezes the PROCEDURE only - NO clean-v3 cases are selected here.')


,item,rule,status,note
0,cohort_size,16,HISTORICAL_VERIFIED,phase11_failure_16_selection.csv contains 16 cases
1,strata,4 categories x 4 cases,HISTORICAL_VERIFIED,Global-Empty; Missed GT Component; Severe FP; Low Dice Decile
2,stratum_1,empty_prediction == True,HISTORICAL_VERIFIED,take first 4 by sample_id
3,stratum_2,missed_eval_components > 0,HISTORICAL_VERIFIED,exclude already-selected; first 4 by sample_id
4,stratum_3,fp_eval_components >= 4,HISTORICAL_AMBIGUOUS,"code uses >=4 but inline comment says "">=5""; COMMENT/CODE mismatch recorded"
5,stratum_4,low_dice_decile == True,HISTORICAL_AMBIGUOUS,"cohort label says ""Low Dice Decile (<0.30)"" but actual cutoff is 0.5461 (bottom decile); LABEL mismatch recorded"
6,tie_break,sort by sample_id (deterministic); low-dice stratum sorts by ascending dice,HISTORICAL_VERIFIED,selection is deterministic and prediction-independent in ordering
7,exclusion,each stratum excludes samples selected by earlier strata,HISTORICAL_VERIFIED,prevents duplicate cases across categories
8,depends_on_prediction_quality,YES by construction (failure strata),HISTORICAL_VERIFIED,"unlike the Phase-10 gallery, this cohort IS selected by failure status - intentional for failure analysis"
9,clean_v3_status,NOT YET APPLIED,FROZEN_PROPOSAL_ONLY,Part 1A freezes the procedure; no clean-v3 cases are selected in Part 1A



Key point: this cohort IS selected by failure status (prediction-dependent), unlike the
Phase-10 qualitative gallery which is prediction-independent. That is intentional for
failure analysis, but it must never be presented as a representative sample.

Part 1A freezes the PROCEDURE only - NO clean-v3 cases are selected here.


## 8. Historical Phase-11 → Phase-12 Rationale


In [8]:
br = p.build_phase12_bridge()
display(br)
print()
print('HISTORICAL RATIONALE (recorded verbatim from the audit):')
print(' >', hist['historical_phase11_to_phase12_rationale'])
print()
print('This is the HISTORICAL justification. It is NOT rewritten from clean-v3 results.')
print('Part 1B must test cleanly whether the same pattern holds across all four clean-v3 models.')


,item,value,source
0,historical_observation,"FULL_FIELD, small-object and multi-component failures persisted for the global baseline",reports/audits/phase11_kub_failure_analysis.json:phase12_motivation
1,historical_rationale,Motivates testing whether increased effective local resolution can recover missed targets; explicitly NOT a claim that ROI will solve them,reports/audits/phase11_kub_failure_analysis.json:phase12_motivation
2,tiny_recall_evidence,"TINY component recall 57/73 = 0.7808 (kub_size_v1, UNet++)",reports/tables/phase11_kub_size_failure_summary.csv
3,multicomponent_evidence,50 multi-component cases: 36 COMPLETE / 10 PARTIAL / 4 COMPLETE_MISS,reports/tables/phase11_kub_multicomponent_failure.csv
4,primary_model_historical,UNet++ (phase10_kub_unetpp_seed42_512),reports/audits/phase11_kub_failure_analysis.json:primary_model
5,comparator_historical,DeepLabV3+,reports/audits/phase11_kub_failure_analysis.json:secondary_comparator
6,clean_v3_reference_model,NOT SELECTED,Part 1A must not choose a reference from single-seed VAL (see spec section 22)
7,part_1b_must_test,"Whether the same FULL_FIELD / small-object / multi-component failure pattern holds for ALL FOUR clean-v3 baselines, and whether it is shared or architecture-specific",derived



HISTORICAL RATIONALE (recorded verbatim from the audit):
 > Observed FULL_FIELD, small-object, and multi-component failures motivate testing whether increased effective local resolution can recover missed targets; no claim that ROI will solve them.

This is the HISTORICAL justification. It is NOT rewritten from clean-v3 results.
Part 1B must test cleanly whether the same pattern holds across all four clean-v3 models.


## 9. Clean-v3 Invariant Contracts


In [9]:
print('=== CLEAN-v3 INVARIANT CONTRACTS ===')
print('Foreground rule        :', p.FOREGROUND_RULE, '(never mask == 1)')
print('Connectivity           :', p.CONNECTIVITY)
print('Minimum overlap pixels :', p.MINIMUM_OVERLAP_PIXELS)
print('Prediction threshold   : 0.5 (fixed, no tuning)')
print('Taxonomy               :', p.TAXONOMY_ID)
print('Taxonomy quantiles     :', p.TAXONOMY_QUANTILES)
print('Taxonomy scope         : FULL_FIELD only; ROI = ROI_NOT_PRIMARY_BINNED')
print('Split                  : KUB VAL, 280 cases')
print()
print('These come from frozen repository contracts:')
print('  - src/utils/targets.py (binarize_mask -> mask > 0)')
print('  - src/metrics/object_metrics.py (8-conn, overlap-based, min 1 px)')
print('  - reports/audits/kub_size_clean_v3_v1_thresholds.json')
assert p.TAXONOMY_SHA == p.sha256_file(p.AUDITS/'kub_size_clean_v3_v1_thresholds.json')
print('\nTaxonomy artifact SHA verified:', p.TAXONOMY_SHA)


=== CLEAN-v3 INVARIANT CONTRACTS ===
Foreground rule        : mask > 0 (never mask == 1)
Connectivity           : 8
Minimum overlap pixels : 1
Prediction threshold   : 0.5 (fixed, no tuning)
Taxonomy               : kub_size_clean_v3_v1
Taxonomy quantiles     : {'Q25': 0.0001497998649691358, 'Q50': 0.0003401438395182292, 'Q75': 0.0008279682677469136}
Taxonomy scope         : FULL_FIELD only; ROI = ROI_NOT_PRIMARY_BINNED
Split                  : KUB VAL, 280 cases

These come from frozen repository contracts:
  - src/utils/targets.py (binarize_mask -> mask > 0)
  - src/metrics/object_metrics.py (8-conn, overlap-based, min 1 px)
  - reports/audits/kub_size_clean_v3_v1_thresholds.json

Taxonomy artifact SHA verified: 89031c51a35c24f6f939fd6206fde6002a019d4bda24ce8cb21988f04db0f0d6


## 10. Acquisition-Regime Contract

FULL_FIELD and ROI are **different acquisition regimes**; object scale differs radically.
They must never be pooled without showing both.


In [10]:
print('Regime counts (clean-v3 KUB VAL):', p.REGIME_COUNTS)
print('Historical (v2)            : ROI_CROP 177 / FULL_FIELD 106')
print()
print('Rule: any failure rate must be reported per regime. A pooled number alone is invalid.')


Regime counts (clean-v3 KUB VAL): {'FULL_FIELD': 103, 'ROI': 177}
Historical (v2)            : ROI_CROP 177 / FULL_FIELD 106

Rule: any failure rate must be reported per regime. A pooled number alone is invalid.


## 11. Size-Taxonomy Contract


In [11]:
print('Taxonomy:', p.TAXONOMY_ID)
for k,v in p.TAXONOMY_QUANTILES.items(): print(f'  {k} = {v}')
print()
print('Bins (lower-inclusive): TINY <= Q25 < SMALL <= Q50 < MEDIUM <= Q75 < LARGE')
print('Scope: FULL_FIELD only. ROI cases are ROI_NOT_PRIMARY_BINNED.')
print('HISTORICAL taxonomy kub_size_v1 is retained for provenance only and must NOT be')
print('applied to clean-v3 data (different quantiles / different population).')


Taxonomy: kub_size_clean_v3_v1
  Q25 = 0.0001497998649691358
  Q50 = 0.0003401438395182292
  Q75 = 0.0008279682677469136

Bins (lower-inclusive): TINY <= Q25 < SMALL <= Q50 < MEDIUM <= Q75 < LARGE
Scope: FULL_FIELD only. ROI cases are ROI_NOT_PRIMARY_BINNED.
HISTORICAL taxonomy kub_size_v1 is retained for provenance only and must NOT be
applied to clean-v3 data (different quantiles / different population).


## 12. Object / Connectivity Contract


In [12]:
print('=== OBJECT CONTRACT ===')
print('Foreground            : mask > 0')
print('Connectivity          : 8 (src/metrics/object_metrics.py)')
print('Matching              : overlap-based, minimum_overlap_pixels = 1')
print('Component space       : original space = identity + size + single/multi grouping')
print('                        evaluation 512 space = prediction components, object recall, FP components')
print()
print('The two spaces must never be silently mixed (resize/padding can merge components).')
print('Connectivity MUST be declared in any object-level output (4- vs 8-connectivity differ).')


=== OBJECT CONTRACT ===
Foreground            : mask > 0
Connectivity          : 8 (src/metrics/object_metrics.py)
Matching              : overlap-based, minimum_overlap_pixels = 1
Component space       : original space = identity + size + single/multi grouping
                        evaluation 512 space = prediction components, object recall, FP components

The two spaces must never be silently mixed (resize/padding can merge components).
Connectivity MUST be declared in any object-level output (4- vs 8-connectivity differ).


## 13. Candidate Clean-v3 Failure Dimensions


In [13]:
dim = p.build_dimensions()
display(dim)
print()
print('VERIFIED_CONTRACT = already frozen by existing repository contracts')
print('HISTORICAL_VERIFIED / _PROPOSED_EXTENSION = inherited or proposed, see status column')
print('CANDIDATE_FOR_SUPERVISOR_REVIEW = NOT frozen; requires a supervisor decision')


,dimension,definition,unit,status,note
0,regime,FULL_FIELD vs ROI acquisition regime,case,VERIFIED_CONTRACT,FULL_FIELD 103 / ROI 177; NEVER pool without showing both
1,category,Kidney/Bladder/Ureteric subgroup metadata,case,VERIFIED_CONTRACT,metadata only; not a pixel class; Ureteric n=24 do not overinterpret
2,size stratum (FULL_FIELD only),kub_size_clean_v3_v1 TINY/SMALL/MEDIUM/LARGE,component,VERIFIED_CONTRACT,Q25/Q50/Q75 frozen; ROI = ROI_NOT_PRIMARY_BINNED
3,object detection,"per-GT-component detected/missed via 8-conn, >=1px overlap",component,VERIFIED_CONTRACT,original space = identity; evaluation 512 space = predictions/FP
4,multi-component,COMPLETE / PARTIAL / COMPLETE_MISS detection classes,case,HISTORICAL_VERIFIED_PROPOSED_EXTENSION,historical analysed on original_component_count>1; retain as descriptive subgroup
5,boundary,HD95 with validity flag; invalid handled explicitly,case,VERIFIED_CONTRACT,no silent NaN dropping; report valid counts
6,empty prediction,zero-foreground prediction,case,HISTORICAL_VERIFIED,carry forward from historical definition
7,false-positive burden,fp evaluation components count,case,HISTORICAL_VERIFIED,historical used decile cutoff; clean-v3 cutoff NOT yet frozen
8,low-Dice stratum,bottom-decile Dice stratum,case,HISTORICAL_METHOD_NOT_VALUE,method (own-distribution bottom decile) is historical; the numeric value must be recomputed on clean-v3 and is NOT frozen here
9,shared failure,cases difficult across all four clean-v3 baselines,case,CANDIDATE_FOR_SUPERVISOR_REVIEW,no formula frozen in Part 1A



VERIFIED_CONTRACT = already frozen by existing repository contracts
HISTORICAL_VERIFIED / _PROPOSED_EXTENSION = inherited or proposed, see status column
CANDIDATE_FOR_SUPERVISOR_REVIEW = NOT frozen; requires a supervisor decision


## 14. Shared-Failure Governance

Historical concept exists: `ALL_FOUR_LOW` (low in all four models, each by its own bottom
Dice decile). Clean-v3 candidate formulas are prepared but **not adopted**.


In [14]:
print('=== SHARED-FAILURE GOVERNANCE ===')
print('Historical concept : ALL_FOUR_LOW (each model own bottom Dice decile)')
print('Historical status  :', 'VERIFIED (historical, v2)')
print()
print('Clean-v3 candidate definitions (CANDIDATE_FOR_SUPERVISOR_REVIEW - NOT adopted):')
for c in ['mean Dice across the four models','median Dice across models',
          'number of models below the (recomputed) low-Dice stratum']:
    print('  -', c)
print()
print('No shared-failure formula is frozen in Part 1A.')


=== SHARED-FAILURE GOVERNANCE ===
Historical concept : ALL_FOUR_LOW (each model own bottom Dice decile)
Historical status  : VERIFIED (historical, v2)

Clean-v3 candidate definitions (CANDIDATE_FOR_SUPERVISOR_REVIEW - NOT adopted):
  - mean Dice across the four models
  - median Dice across models
  - number of models below the (recomputed) low-Dice stratum

No shared-failure formula is frozen in Part 1A.


## 15. Architecture-Disagreement Governance


In [15]:
print('=== ARCHITECTURE-DISAGREEMENT GOVERNANCE ===')
print('Historical concept : ARCHITECTURE_DISAGREEMENT / *_UNIQUELY_* categories')
print('Historical status  : VERIFIED (historical, v2)')
print()
print('Clean-v3 candidate measures (CANDIDATE_FOR_SUPERVISOR_REVIEW - NOT adopted):')
for c in ['max Dice - min Dice','standard deviation of per-model Dice']:
    print('  -', c)
print()
print('No disagreement threshold is frozen in Part 1A.')


=== ARCHITECTURE-DISAGREEMENT GOVERNANCE ===
Historical concept : ARCHITECTURE_DISAGREEMENT / *_UNIQUELY_* categories
Historical status  : VERIFIED (historical, v2)

Clean-v3 candidate measures (CANDIDATE_FOR_SUPERVISOR_REVIEW - NOT adopted):
  - max Dice - min Dice
  - standard deviation of per-model Dice

No disagreement threshold is frozen in Part 1A.


## 16. Hard-Case-Selection Freeze Proposal

Selection procedure is frozen here; **no cases are selected** until Part 1B.


In [16]:
print('=== HARD-CASE SELECTION FREEZE (proposal) ===')
print('Selection metric : failure-stratum membership (historical 4-strata family)')
print('Cohort size      : 16 (4 strata x 4 cases) - historical precedent')
print('Stratification   : by failure stratum AND acquisition regime')
print('Tie-break        : deterministic sort by sample_id (low-dice stratum: ascending dice)')
print('Exclusion        : later strata exclude earlier-selected samples')
print()
print('STATUS: PROCEDURE_FROZEN_NO_CASES_SELECTED')
print()
print('Governance: the selector is defined BEFORE looking at qualitative outputs, so no')
print('cherry-picking is possible. Selection depends on failure status by design.')


=== HARD-CASE SELECTION FREEZE (proposal) ===
Selection metric : failure-stratum membership (historical 4-strata family)
Cohort size      : 16 (4 strata x 4 cases) - historical precedent
Stratification   : by failure stratum AND acquisition regime
Tie-break        : deterministic sort by sample_id (low-dice stratum: ascending dice)
Exclusion        : later strata exclude earlier-selected samples

STATUS: PROCEDURE_FROZEN_NO_CASES_SELECTED

Governance: the selector is defined BEFORE looking at qualitative outputs, so no
cherry-picking is possible. Selection depends on failure status by design.


## 17. Open Methodological Questions (returned to supervisor)

These require a scientific decision and are **not** resolved here.


In [17]:
qs = [
 ('Low-Dice stratum numeric value',
  'Historical cutoff 0.5461 is v2-specific. Clean-v3 value must be recomputed on clean-v3 VAL. Should the METHOD (bottom decile) be adopted, with value recomputed?'),
 ('Shared-failure formula',
  'Which definition (mean / median / count-below-stratum) should be adopted for clean-v3?'),
 ('Disagreement measure + threshold',
  'max-min Dice or SD, and whether any threshold may be set?'),
 ('FP-burden threshold',
  'Historical used >=4 fp components (comment said >=5). Which for clean-v3?'),
 ('Pixel error decomposition',
  'TP/FP/FN and under/over-segmentation were NOT historically defined - adopt as extension?'),
 ('Multi-component subgroup',
  'Historically analysed; retain as descriptive clean-v3 subgroup? (PROPOSED_EXTENSION)'),
 ('Reference model for Phase 12',
  'Historical was UNet++-primary. Clean-v3 reference must be a separate supervisor decision.'),
]
for i,(q,d) in enumerate(qs,1): print(f'{i}. {q}\n   {d}\n')
print('None of these are decided in Part 1A.')


1. Low-Dice stratum numeric value
   Historical cutoff 0.5461 is v2-specific. Clean-v3 value must be recomputed on clean-v3 VAL. Should the METHOD (bottom decile) be adopted, with value recomputed?

2. Shared-failure formula
   Which definition (mean / median / count-below-stratum) should be adopted for clean-v3?

3. Disagreement measure + threshold
   max-min Dice or SD, and whether any threshold may be set?

4. FP-burden threshold
   Historical used >=4 fp components (comment said >=5). Which for clean-v3?

5. Pixel error decomposition
   TP/FP/FN and under/over-segmentation were NOT historically defined - adopt as extension?

6. Multi-component subgroup
   Historically analysed; retain as descriptive clean-v3 subgroup? (PROPOSED_EXTENSION)

7. Reference model for Phase 12
   Historical was UNet++-primary. Clean-v3 reference must be a separate supervisor decision.

None of these are decided in Part 1A.


## 18. Upstream Complete Epoch Table — UNet

Read-only, from the frozen Phase-10 normalized epoch table. **Every epoch row is shown.**


In [18]:
t = p.load_upstream_epoch_table('unet')
v = p.verify_upstream_epoch_tables().query("model == 'UNet'").iloc[0]
print('=== UPSTREAM INTEGRITY: UNet ===')
for kk in ['rows','expected_rows','row_count_matches','first_epoch','last_epoch',
           'missing_epochs','duplicate_epochs','contiguous']:
    print(f'  {kk}: {v[kk]}')
print()
assert v['row_count_matches'] and v['contiguous']
print(f"Rendering ALL {len(t)} epoch rows for UNet:")
display(t)


=== UPSTREAM INTEGRITY: UNet ===
  rows: 59
  expected_rows: 59
  row_count_matches: True
  first_epoch: 1
  last_epoch: 59
  missing_epochs: []
  duplicate_epochs: []
  contiguous: True

Rendering ALL 59 epoch rows for UNet:


,model,experiment_id,epoch,train_loss,val_loss,val_macro_dice,val_macro_iou,val_precision,val_recall,learning_rate,best_metric_so_far,improved,bad_epochs,elapsed_seconds,checkpoint_event,delta_val_dice_vs_previous_epoch,delta_best_metric,is_new_best,is_best_epoch,epochs_since_best
0,UNet,phase9_kub_unet_baseline_seed42_512_clean_v3,1,0.442458,0.465926,0.431176,0.340278,NaN,NaN,0.000100,NaN,1,0,268.742252,BEST_SAVED,NaN,NaN,True,False,0
1,UNet,phase9_kub_unet_baseline_seed42_512_clean_v3,2,0.392092,0.396965,0.456776,0.366742,NaN,NaN,0.000100,NaN,1,0,60.102438,BEST_SAVED,0.025599,NaN,True,False,0
2,UNet,phase9_kub_unet_baseline_seed42_512_clean_v3,3,0.367363,0.359159,0.490891,0.401569,NaN,NaN,0.000100,NaN,1,0,60.311679,BEST_SAVED,0.034116,NaN,True,False,0
3,UNet,phase9_kub_unet_baseline_seed42_512_clean_v3,4,0.351135,0.425429,0.497879,0.397826,NaN,NaN,0.000100,NaN,1,0,60.858533,BEST_SAVED,0.006987,NaN,True,False,0
4,UNet,phase9_kub_unet_baseline_seed42_512_clean_v3,5,0.335718,0.332848,0.530259,0.438920,NaN,NaN,0.000099,NaN,1,0,63.602900,BEST_SAVED,0.032380,NaN,True,False,0
5,UNet,phase9_kub_unet_baseline_seed42_512_clean_v3,6,0.321234,0.321200,0.545095,0.452030,NaN,NaN,0.000099,NaN,1,0,63.525723,BEST_SAVED,0.014836,NaN,True,False,0
6,UNet,phase9_kub_unet_baseline_seed42_512_clean_v3,7,0.306985,0.305672,0.579637,0.472824,NaN,NaN,0.000099,NaN,1,0,64.249523,BEST_SAVED,0.034542,NaN,True,False,0
7,UNet,phase9_kub_unet_baseline_seed42_512_clean_v3,8,0.292267,0.391134,0.563304,0.445191,NaN,NaN,0.000098,NaN,0,1,63.671509,LATEST_ONLY,-0.016333,NaN,False,False,1
8,UNet,phase9_kub_unet_baseline_seed42_512_clean_v3,9,0.272067,0.287922,0.638627,0.520392,NaN,NaN,0.000098,NaN,1,0,63.566675,BEST_SAVED,0.075323,NaN,True,False,0
9,UNet,phase9_kub_unet_baseline_seed42_512_clean_v3,10,0.261210,0.260292,0.654405,0.541183,NaN,NaN,0.000098,NaN,1,0,63.609702,BEST_SAVED,0.015778,NaN,True,False,0


## 19. Upstream Complete Epoch Table — Attention U-Net

Read-only, from the frozen Phase-10 normalized epoch table. **Every epoch row is shown.**


In [19]:
t = p.load_upstream_epoch_table('attention_unet')
v = p.verify_upstream_epoch_tables().query("model == 'Attention U-Net'").iloc[0]
print('=== UPSTREAM INTEGRITY: Attention U-Net ===')
for kk in ['rows','expected_rows','row_count_matches','first_epoch','last_epoch',
           'missing_epochs','duplicate_epochs','contiguous']:
    print(f'  {kk}: {v[kk]}')
print()
assert v['row_count_matches'] and v['contiguous']
print(f"Rendering ALL {len(t)} epoch rows for Attention U-Net:")
display(t)


=== UPSTREAM INTEGRITY: Attention U-Net ===
  rows: 69
  expected_rows: 69
  row_count_matches: True
  first_epoch: 1
  last_epoch: 69
  missing_epochs: []
  duplicate_epochs: []
  contiguous: True

Rendering ALL 69 epoch rows for Attention U-Net:


,model,experiment_id,epoch,train_loss,val_loss,val_macro_dice,val_macro_iou,val_precision,val_recall,learning_rate,best_metric_so_far,improved,bad_epochs,elapsed_seconds,checkpoint_event,delta_val_dice_vs_previous_epoch,delta_best_metric,is_new_best,is_best_epoch,epochs_since_best
0,Attention U-Net,phase10_kub_attention_unet_seed42_512_clean_v3,1,0.511852,0.492498,0.455501,0.364793,0.557557,0.601329,0.000100,0.455501,1,0,180.8,BEST_SAVED,NaN,NaN,True,False,0
1,Attention U-Net,phase10_kub_attention_unet_seed42_512_clean_v3,2,0.421238,0.414937,0.453676,0.365484,0.583015,0.572629,0.000100,0.455501,0,1,73.8,LATEST_ONLY,-0.001825,0.000000,False,False,1
2,Attention U-Net,phase10_kub_attention_unet_seed42_512_clean_v3,3,0.385591,0.402041,0.469479,0.383368,0.685439,0.592345,0.000100,0.469479,1,0,70.7,BEST_SAVED,0.015802,0.013977,True,False,0
3,Attention U-Net,phase10_kub_attention_unet_seed42_512_clean_v3,4,0.363607,0.365701,0.492169,0.400162,0.595332,0.512817,0.000100,0.492169,1,0,70.3,BEST_SAVED,0.022690,0.022690,True,False,0
4,Attention U-Net,phase10_kub_attention_unet_seed42_512_clean_v3,5,0.347567,0.357465,0.507544,0.418799,0.459013,0.673806,0.000099,0.507544,1,0,73.2,BEST_SAVED,0.015375,0.015375,True,False,0
5,Attention U-Net,phase10_kub_attention_unet_seed42_512_clean_v3,6,0.335710,0.337032,0.542214,0.445455,0.538902,0.645993,0.000099,0.542214,1,0,73.1,BEST_SAVED,0.034670,0.034670,True,False,0
6,Attention U-Net,phase10_kub_attention_unet_seed42_512_clean_v3,7,0.326942,0.326772,0.546270,0.452797,0.531451,0.653997,0.000099,0.546270,1,0,73.4,BEST_SAVED,0.004056,0.004056,True,False,0
7,Attention U-Net,phase10_kub_attention_unet_seed42_512_clean_v3,8,0.313945,0.307033,0.566118,0.473826,0.656369,0.611473,0.000098,0.566118,1,0,74.5,BEST_SAVED,0.019848,0.019848,True,False,0
8,Attention U-Net,phase10_kub_attention_unet_seed42_512_clean_v3,9,0.299837,0.293197,0.603426,0.499475,0.653554,0.661779,0.000098,0.603426,1,0,74.2,BEST_SAVED,0.037307,0.037307,True,False,0
9,Attention U-Net,phase10_kub_attention_unet_seed42_512_clean_v3,10,0.285437,0.288194,0.604399,0.499341,0.690762,0.631612,0.000098,0.604399,1,0,74.3,BEST_SAVED,0.000973,0.000973,True,False,0


## 20. Upstream Complete Epoch Table — UNet++

Read-only, from the frozen Phase-10 normalized epoch table. **Every epoch row is shown.**


In [20]:
t = p.load_upstream_epoch_table('unetpp')
v = p.verify_upstream_epoch_tables().query("model == 'UNet++'").iloc[0]
print('=== UPSTREAM INTEGRITY: UNet++ ===')
for kk in ['rows','expected_rows','row_count_matches','first_epoch','last_epoch',
           'missing_epochs','duplicate_epochs','contiguous']:
    print(f'  {kk}: {v[kk]}')
print()
assert v['row_count_matches'] and v['contiguous']
print(f"Rendering ALL {len(t)} epoch rows for UNet++:")
display(t)


=== UPSTREAM INTEGRITY: UNet++ ===
  rows: 85
  expected_rows: 85
  row_count_matches: True
  first_epoch: 1
  last_epoch: 85
  missing_epochs: []
  duplicate_epochs: []
  contiguous: True

Rendering ALL 85 epoch rows for UNet++:


,model,experiment_id,epoch,train_loss,val_loss,val_macro_dice,val_macro_iou,val_precision,val_recall,learning_rate,best_metric_so_far,improved,bad_epochs,elapsed_seconds,checkpoint_event,delta_val_dice_vs_previous_epoch,delta_best_metric,is_new_best,is_best_epoch,epochs_since_best
0,UNet++,phase10_kub_unetpp_seed42_512_clean_v3,1,0.438780,0.416830,0.454545,0.363090,0.678880,0.502299,0.000100,0.454545,1,0,290.9,BEST_SAVED,NaN,NaN,True,False,0
1,UNet++,phase10_kub_unetpp_seed42_512_clean_v3,2,0.388098,0.421986,0.468632,0.378370,0.704706,0.592291,0.000100,0.468632,1,0,180.4,BEST_SAVED,0.014087,0.014087,True,False,0
2,UNet++,phase10_kub_unetpp_seed42_512_clean_v3,3,0.363468,0.443350,0.465154,0.372904,0.571621,0.639822,0.000100,0.468632,0,1,180.6,LATEST_ONLY,-0.003477,0.000000,False,False,1
3,UNet++,phase10_kub_unetpp_seed42_512_clean_v3,4,0.352005,0.348390,0.519276,0.427362,0.499869,0.639654,0.000100,0.519276,1,0,180.6,BEST_SAVED,0.054121,0.050644,True,False,0
4,UNet++,phase10_kub_unetpp_seed42_512_clean_v3,5,0.336932,0.323569,0.554764,0.460919,0.562959,0.635015,0.000099,0.554764,1,0,178.6,BEST_SAVED,0.035489,0.035489,True,False,0
5,UNet++,phase10_kub_unetpp_seed42_512_clean_v3,6,0.325398,0.328325,0.562319,0.462126,0.589963,0.666141,0.000099,0.562319,1,0,178.7,BEST_SAVED,0.007554,0.007554,True,False,0
6,UNet++,phase10_kub_unetpp_seed42_512_clean_v3,7,0.310921,0.328777,0.551526,0.443353,0.545447,0.666942,0.000099,0.562319,0,1,178.6,LATEST_ONLY,-0.010793,0.000000,False,False,1
7,UNet++,phase10_kub_unetpp_seed42_512_clean_v3,8,0.297655,0.320062,0.547991,0.450617,0.499140,0.808599,0.000098,0.562319,0,2,178.9,LATEST_ONLY,-0.003535,0.000000,False,False,2
8,UNet++,phase10_kub_unetpp_seed42_512_clean_v3,9,0.265427,0.276223,0.620099,0.504560,0.570966,0.801323,0.000098,0.620099,1,0,178.9,BEST_SAVED,0.072109,0.057781,True,False,0
9,UNet++,phase10_kub_unetpp_seed42_512_clean_v3,10,0.258951,0.272386,0.659271,0.536225,0.616674,0.808719,0.000098,0.659271,1,0,179.0,BEST_SAVED,0.039172,0.039172,True,False,0


## 21. Upstream Complete Epoch Table — DeepLabV3+

Read-only, from the frozen Phase-10 normalized epoch table. **Every epoch row is shown.**


In [21]:
t = p.load_upstream_epoch_table('deeplabv3plus')
v = p.verify_upstream_epoch_tables().query("model == 'DeepLabV3+'").iloc[0]
print('=== UPSTREAM INTEGRITY: DeepLabV3+ ===')
for kk in ['rows','expected_rows','row_count_matches','first_epoch','last_epoch',
           'missing_epochs','duplicate_epochs','contiguous']:
    print(f'  {kk}: {v[kk]}')
print()
assert v['row_count_matches'] and v['contiguous']
print(f"Rendering ALL {len(t)} epoch rows for DeepLabV3+:")
display(t)


=== UPSTREAM INTEGRITY: DeepLabV3+ ===
  rows: 67
  expected_rows: 67
  row_count_matches: True
  first_epoch: 1
  last_epoch: 67
  missing_epochs: []
  duplicate_epochs: []
  contiguous: True

Rendering ALL 67 epoch rows for DeepLabV3+:


,model,experiment_id,epoch,train_loss,val_loss,val_macro_dice,val_macro_iou,val_precision,val_recall,learning_rate,best_metric_so_far,improved,bad_epochs,elapsed_seconds,checkpoint_event,delta_val_dice_vs_previous_epoch,delta_best_metric,is_new_best,is_best_epoch,epochs_since_best
0,DeepLabV3+,phase10_kub_deeplabv3plus_seed42_512_clean_v3,1,0.377359,0.349518,0.487975,0.407391,0.870639,0.504660,0.000100,0.487975,1,0,188.1,BEST_SAVED,NaN,NaN,True,False,0
1,DeepLabV3+,phase10_kub_deeplabv3plus_seed42_512_clean_v3,2,0.330579,0.314964,0.540444,0.460717,0.769889,0.578849,0.000100,0.540444,1,0,74.1,BEST_SAVED,0.052469,0.052469,True,False,0
2,DeepLabV3+,phase10_kub_deeplabv3plus_seed42_512_clean_v3,3,0.307240,0.299174,0.563059,0.480463,0.749197,0.596709,0.000100,0.563059,1,0,73.8,BEST_SAVED,0.022615,0.022615,True,False,0
3,DeepLabV3+,phase10_kub_deeplabv3plus_seed42_512_clean_v3,4,0.287705,0.291801,0.574313,0.487857,0.736589,0.615886,0.000100,0.574313,1,0,74.1,BEST_SAVED,0.011254,0.011254,True,False,0
4,DeepLabV3+,phase10_kub_deeplabv3plus_seed42_512_clean_v3,5,0.273234,0.300474,0.571880,0.480890,0.696173,0.606999,0.000099,0.574313,0,1,73.4,LATEST_ONLY,-0.002432,0.000000,False,False,1
5,DeepLabV3+,phase10_kub_deeplabv3plus_seed42_512_clean_v3,6,0.261933,0.269859,0.602290,0.513627,0.702040,0.631256,0.000099,0.602290,1,0,73.4,BEST_SAVED,0.030410,0.027978,True,False,0
6,DeepLabV3+,phase10_kub_deeplabv3plus_seed42_512_clean_v3,7,0.246003,0.270271,0.610485,0.516051,0.617278,0.741697,0.000099,0.610485,1,0,73.3,BEST_SAVED,0.008194,0.008194,True,False,0
7,DeepLabV3+,phase10_kub_deeplabv3plus_seed42_512_clean_v3,8,0.233239,0.248815,0.645464,0.541690,0.721210,0.702844,0.000098,0.645464,1,0,74.8,BEST_SAVED,0.034979,0.034979,True,False,0
8,DeepLabV3+,phase10_kub_deeplabv3plus_seed42_512_clean_v3,9,0.221540,0.234570,0.669442,0.563907,0.757588,0.721188,0.000098,0.669442,1,0,73.4,BEST_SAVED,0.023978,0.023978,True,False,0
9,DeepLabV3+,phase10_kub_deeplabv3plus_seed42_512_clean_v3,10,0.213646,0.225732,0.683498,0.576951,0.709659,0.729663,0.000098,0.683498,1,0,73.2,BEST_SAVED,0.014057,0.014057,True,False,0


## 22. Proposed Part-1B Execution Plan

Part 1B (not started) should:
1. Apply the frozen definitions to the four clean-v3 baselines using the existing paired table.
2. Report failure rates **per regime** (FULL_FIELD 103 / ROI 177) and per category.
3. Report object-level detection using 8-conn / >=1 px, in declared component space.
4. Compute the shared-failure and disagreement views **once the supervisor picks a formula**.
5. Select the hard-case cohort using the frozen procedure only after the above.
6. Preserve full epoch-table visibility for any newly created notebook.


## 23. Artifact Hashes


In [22]:
def _sh(rel):
    q = PROJECT_ROOT / rel
    return p.sha256_file(q) if q.exists() else 'N/A'
arts = [
 ('notebooks/phasewise/phase_11_failure_analysis/code/phase11_failure_protocol.py',),
 ('reports/tables/phase11_historical_source_inventory.csv',),
 ('reports/tables/phase11_historical_failure_definition_matrix.csv',),
 ('reports/tables/phase11_clean_v3_failure_analysis_dimensions.csv',),
 ('reports/tables/phase11_clean_v3_hard_case_selection_protocol.csv',),
 ('reports/tables/phase11_phase12_bridge_reconstruction.csv',),
 ('reports/audits/phase11_preflight_state_consistency_audit.json',),
 ('reports/audits/phase11_historical_failure_protocol_reconstruction.json',),
 ('reports/audits/phase11_clean_v3_failure_analysis_protocol_candidate.json',),
 ('reports/audits/antigravity_current_state.json',),
]
REG = pd.DataFrame([{'artifact': Path(a[0]).name, 'path': a[0], 'sha256': _sh(a[0])} for a in arts])
rp = p.TABLES / 'phase11_part1a_artifact_registry.csv'
REG.to_csv(rp, index=False)
print(f'Saved registry: {{rp.name}}')
display(REG)


Saved registry: {rp.name}


,artifact,path,sha256
0,phase11_failure_protocol.py,notebooks/phasewise/phase_11_failure_analysis/code/phase11_failure_protocol.py,b6c16e02b72764dd34752cebff098d10f55f0c0432d38727a637cc0c61622607
1,phase11_historical_source_inventory.csv,reports/tables/phase11_historical_source_inventory.csv,fc10b15e03e04a7674619858392c863628c945b568f3b99241f7652e9fc8bdcd
2,phase11_historical_failure_definition_matrix.csv,reports/tables/phase11_historical_failure_definition_matrix.csv,dc4fbf3a8601ab018076a83bbb2f5be3d5f4f11c11ba356d94f76ad52b6ffe4a
3,phase11_clean_v3_failure_analysis_dimensions.csv,reports/tables/phase11_clean_v3_failure_analysis_dimensions.csv,0bbecc6bbe01672392bcc7bfc94895e92567a4068e77e43846c9b62a7e9a9ae4
4,phase11_clean_v3_hard_case_selection_protocol.csv,reports/tables/phase11_clean_v3_hard_case_selection_protocol.csv,7dfd784f06c1083d498c44d9ba6a81346f479c5618b58500204ff758f5a65c5a
5,phase11_phase12_bridge_reconstruction.csv,reports/tables/phase11_phase12_bridge_reconstruction.csv,d27988f6d5b75cd0e4b82187282c4a3d8da5fd6d2199b6b73bc8d49dc59c66dc
6,phase11_preflight_state_consistency_audit.json,reports/audits/phase11_preflight_state_consistency_audit.json,4ca968cfa5bbff21eb28dd445d1b63c0363775a2b4ebda5fecfc4e5ea80927c9
7,phase11_historical_failure_protocol_reconstruction.json,reports/audits/phase11_historical_failure_protocol_reconstruction.json,b928eb8f0309503632628879926d397bf7bf3027a439e27603009d19f7559f51
8,phase11_clean_v3_failure_analysis_protocol_candidate.json,reports/audits/phase11_clean_v3_failure_analysis_protocol_candidate.json,92dd18ae84991f25850efb4726bee7d398f26633c934a716f5a369c306a6c9ad
9,antigravity_current_state.json,reports/audits/antigravity_current_state.json,18112bf5fd2302c675a248e5e62dc8fe8b91bb3d5aabaa14bd62d17df8cc4dfc


## 24. Stop Boundary


In [23]:
print('=== PHASE-11 PART-1A BOUNDARY ===')
print('Training epochs                 : 0')
print('Optimizer steps                 : 0')
print('Checkpoint writes               : 0')
print('Histories / configs modified    : 0')
print('TEST image/mask/content reads   : 0')
print('TEST metrics / visualizations   : 0')
print('New unauthorized thresholds     : 0')
print('Winner declared                 : NO')
print('Reference model selected        : NO')
print('Full Phase-11 analysis started  : NO')
print('Phase 12 started                : NO')
print()
print('FAILURE PROTOCOL STATUS: READY_FOR_SUPERVISOR_FREEZE')
print('NEXT: WAIT_FOR_SUPERVISOR_PHASE11_PROTOCOL_FREEZE_DECISION')


=== PHASE-11 PART-1A BOUNDARY ===
Training epochs                 : 0
Optimizer steps                 : 0
Checkpoint writes               : 0
Histories / configs modified    : 0
TEST image/mask/content reads   : 0
TEST metrics / visualizations   : 0
New unauthorized thresholds     : 0
Winner declared                 : NO
Reference model selected        : NO
Full Phase-11 analysis started  : NO
Phase 12 started                : NO

FAILURE PROTOCOL STATUS: READY_FOR_SUPERVISOR_FREEZE
NEXT: WAIT_FOR_SUPERVISOR_PHASE11_PROTOCOL_FREEZE_DECISION
